# LAB: SVM on Cats Dataset

## Objective
- Apply Support Vector Machine (SVM) to classify a dataset.
- Compare the performance of Linear, Polynomial, and RBF kernels.
- Explore preprocessing, clustering, classification, standardization, accuracy, and predictions.

### Dataset
This lab uses `cats(1).csv`. The classification target is **Gender** (`Male` or `Female`) using:
- Age
- Size
- Breed


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('cats(1).csv', header=None)
df.columns = ['RecordID', 'PetID', 'URL', 'AnimalType', 'Age',
              'Gender', 'Size', 'Unnamed', 'Breed',
              'PhotoData', 'PhotoURLs']

df.head()

## 1. Explore the Dataset

In [ ]:
print('Dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

print('\nAge distribution:')
print(df['Age'].value_counts())

print('\nGender distribution:')
print(df['Gender'].value_counts())

print('\nSize distribution:')
print(df['Size'].value_counts())

print('\nBreed distribution:')
print(df['Breed'].value_counts())

In [ ]:
df['Gender'].value_counts().plot(kind='bar')
plt.title('Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

## 2. Select Features and Preprocess the Data

In [ ]:
data = df[['Age', 'Gender', 'Size', 'Breed']].copy()
data = data.dropna()

X = data[['Age', 'Size', 'Breed']]
y = data['Gender']

print('X shape:', X.shape)
print('Target classes:', y.unique())

## 3. Standardization and Data Preparation

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_features = ['Age', 'Size', 'Breed']

preprocessor = ColumnTransformer([
    ('categorical',
     Pipeline([
         ('onehot', OneHotEncoder(handle_unknown='ignore')),
         ('scaler', StandardScaler(with_mean=False))
     ]),
     categorical_features)
])

## 4. Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X_encoded = OneHotEncoder(handle_unknown='ignore').fit_transform(X).toarray()
X_scaled = StandardScaler().fit_transform(X_encoded)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters)
plt.title('K-Means Clustering of Cats Dataset')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

## 5. Split Data for Classification

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 6. Train SVM Models with Different Kernels

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

kernels = ['linear', 'poly', 'rbf']
results = {}
models = {}

for kernel in kernels:
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('svm', SVC(kernel=kernel))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    models[kernel] = model
    results[kernel] = accuracy

    print(f'{kernel.capitalize()} Kernel Accuracy: {accuracy:.4f}')

## 7. Compare Accuracy

In [ ]:
results_df = pd.DataFrame({
    'Kernel': list(results.keys()),
    'Accuracy': list(results.values())
})

results_df

In [ ]:
plt.bar(results_df['Kernel'], results_df['Accuracy'])
plt.title('SVM Kernel Accuracy Comparison')
plt.xlabel('Kernel')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.show()

## 8. Predictions

In [ ]:
prediction_model = models['linear']

sample_data = pd.DataFrame({
    'Age': ['Adult', 'Young', 'Senior', 'Baby'],
    'Size': ['Medium', 'Small', 'Large', 'Small'],
    'Breed': ['Persian', 'Mixed Breed', 'Domestic Short Hair']
})

sample_data = sample_data.iloc[:3]

predictions = prediction_model.predict(sample_data)

prediction_result = sample_data.copy()
prediction_result['Predicted Gender'] = predictions

prediction_result

## 9. Conclusion

The SVM models were trained using three different kernels:

- **Linear Kernel**
- **Polynomial Kernel**
- **RBF Kernel**

The models were evaluated using **accuracy** on the test dataset. The kernel with the highest accuracy can be selected as the best-performing SVM model for this dataset.

Because the selected input features are categorical and have limited variation, the classification performance may be moderate. Different features or a larger dataset could improve the model's ability to predict gender.
